In [2]:
# Pipeline riemanniana a banco di filtri con riallineamento per run, su tutte e sei le run di
# motor imagery: riposo vs attivazione. E' il primo stadio della classificazione a due livelli:
# qui si decide soltanto SE il soggetto sta immaginando un movimento, non quale. Copia di
# RiemannFB_motor_imagery con le sole modifiche necessarie al cambio di problema.
#
# Le run sono sei e non tre perche' al primo stadio il movimento immaginato non conta: le run
# 4, 8, 12 (sinistra vs destra) e le 6, 10, 14 (pugni vs piedi) rispondono alla stessa domanda.
# Raddoppiare le run raddoppia la classe scarsa, che e' l'attivazione: il riposo era gia' in
# eccesso. In piu' il modello impara cosa hanno in comune le attivazioni invece della firma
# specifica della mano, che e' quello che serve a un rilevatore di intenzione generale.
# Le run di motor execution restano fuori di proposito: il movimento reale porta con se'
# attivita' muscolare, e un rilevatore addestrato anche su quelle imparerebbe in buona parte a
# riconoscere i muscoli invece della corteccia.
#
# Perche' l'approccio riemanniano e non il CSP: riposo contro attivazione non e' un contrasto
# spaziale ma una differenza di livello, perche' il ritmo mu cala su entrambi gli emisferi.
# Le feature del CSP sono log-potenze assolute e per riconoscere "meno potenza del solito"
# manca loro il riferimento a cosa sia il solito, che varia molto fra soggetti e fra sessioni.
# Il riallineamento per run costruisce esattamente quel riferimento: dopo la ricentratura ogni
# finestra e' descritta rispetto al comportamento medio della sua stessa run. E dato che due
# terzi delle finestre di una run sono riposo, il baricentro coincide di fatto con lo stato di
# riposo, quindi le finestre attive sono quelle che se ne allontanano.
# --- Preambolo standard ------------------------------------------------------
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "dataset_description.json").exists()
)
DATA  = PROJECT_ROOT / "data"
PLOTS = PROJECT_ROOT / "src" / "plots"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# -----------------------------------------------------------------------------

import time
import warnings
import numpy as np
import mne
from mne_bids import BIDSPath, read_raw_bids
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
from util.preprocessing import create_sliding_windows, create_window_labels
from util.riemann_filterbank import (
    DEFAULT_BANDS, run_covariances, reference_mean, recenter, to_tangent, n_features
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
mne.set_log_level('WARNING')

# --- Configurazione ----------------------------------------------------------

root = DATA
runs = ["4", "8", "12", "6", "10", "14"]
test_run_order = list(reversed(runs))   # ordine in cui le run vengono usate come test set

# Il prefisso dell'annotazione dipende dal compito della run, non dal riposo: le run
# sinistra/destra usano TASK2, quelle pugni/piedi TASK4. Il codice T0 resta il riposo in
# entrambe, T1 e T2 le due classi attive, che qui vengono unite.
TASK_PREFIX = {"4": "TASK2", "8": "TASK2", "12": "TASK2",
               "6": "TASK4", "10": "TASK4", "14": "TASK4"}
first_person = 1
people = 109

window_size = 2         # identici alle altre pipeline, per confrontabilita'
step_size = 0.5
label_threshold = 0.8

# Banda larga: le sotto-bande del banco fanno la selezione fine
L_FREQ = 4
H_FREQ = 40
BANDS = DEFAULT_BANDS   # [(8, 13), (13, 20), (20, 30)]

CHANNELS = ["C3", "C4", "Cz", "Fc3", "Fc4", "Fcz", "Cp3", "Cp4", "Cpz"]

# Le finestre a cavallo fra un trial attivo e il riposo successivo non raggiungono l'80% di
# nessuna classe e vengono etichettate riposo per esclusione, pur contenendo movimento
# immaginato. Nella pipeline sinistra/destra finivano fra gli scarti insieme al riposo; qui il
# riposo e' una classe, quindi se restassero ne inquinerebbero le etichette.
DROP_BOUNDARY = True

# Riallineamento delle covarianze:
#   "per_run" -> ogni run viene riportata sul proprio baricentro. Il baricentro non usa le
#                etichette, quindi si puo' stimare anche sulla run di test: e' adattamento di
#                dominio non supervisionato. E' la configurazione che ci si aspetta migliore.
#   "train"   -> tutte le run vengono riportate sul baricentro delle sole run di training.
#                Piu' conservativo: nessuna statistica della run di test viene usata.
RECENTER = "per_run"

# Il classificatore e' quasi privo di iperparametri: la LDA con shrinkage di Ledoit-Wolf
# regolarizza da sola ed e' deterministica, a differenza dell'SVM con probability=True le cui
# probabilita' passano da una calibrazione di Platt con mescolamento casuale interno.
# Griglia minuscola: con 28 trial indipendenti cercare fra decine di combinazioni insegue rumore.
param_grid = {
    "clf__shrinkage": ["auto", 0.2, 0.5],
}

START_THRESHOLD = 0.90
MIN_THRESHOLD = 0.50
MIN_ACCEPTED_RATIO = 0.70

# NaN e non 0: un fold che fallisce deve restare fuori dalle medie, non entrarci come 0%
all_accuracy = np.full((people, len(test_run_order)), np.nan)
all_discarded = np.full((people, len(test_run_order)), np.nan)
cm_sum = np.zeros((2, 2))

print(f"Feature per finestra: {n_features(len(CHANNELS), len(BANDS))} "
      f"({len(BANDS)} bande x {len(CHANNELS)} canali)")
print()

# --- Loop principale ---------------------------------------------------------

for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    subject_start = time.time()

    print("=" * 60)
    print(f"Paziente {subject}")
    print("=" * 60)

    # Lettura, finestre e covarianze dipendono solo dal segnale, non da quale run faccia da
    # test: si calcolano una volta sola per soggetto invece di ripeterle dentro ogni fold
    # della LORO. Per ogni run si conservano le covarianze di TUTTE le finestre (servono a
    # stimare il baricentro senza usare le etichette) e la maschera di quelle utilizzabili.
    per_run = {}
    trial_offset = 0

    for run in runs:
        bids_path = BIDSPath(
            subject=subject, task="motion", run=run, datatype="eeg", root=root,
        )

        try:
            raw = read_raw_bids(bids_path, verbose=False)
            events, event_id = mne.events_from_annotations(raw, verbose=False)
            raw.load_data(verbose=False)
            raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)
            raw.set_eeg_reference('average', projection=False, verbose=False)

            sfreq = raw.info['sfreq']
            prefix = TASK_PREFIX[run]
            event_map = {
                event_id[f'{prefix}T0']: 1,
                event_id[f'{prefix}T1']: 2,
                event_id[f'{prefix}T2']: 3
            }

            windows, window_samples, step_samples, total_samples = create_sliding_windows(
                raw, window_size, step_size
            )
            picks = mne.pick_channels(raw.ch_names, CHANNELS)
            windows = windows[:, picks, :]

            y, groups = create_window_labels(
                events, event_map, total_samples, window_samples, step_samples,
                threshold=label_threshold, return_groups=True
            )
            groups = np.where(groups >= 0, groups + trial_offset, -1)
            trial_offset += len(events)

            # Covarianze su TUTTE le finestre: il baricentro va stimato sull'intera run,
            # senza guardare quali finestre siano attive ne' di che classe siano.
            covs = run_covariances(windows, sfreq, BANDS)

            # Anche il baricentro di ogni run e' indipendente dal fold, quindi si calcola qui.
            # Con RECENTER = "train" il riferimento dipende invece da quali run siano di
            # training e viene ricalcolato dentro il ciclo dei fold.
            refs = [reference_mean(c) for c in covs] if RECENTER == "per_run" else None

            # Riposo (0) contro attivazione (1), senza distinguere sinistra e destra.
            y_binary = np.where(y == 1, 0, 1)

            # Le finestre col sentinella -1 cadono dopo l'ultimo evento della run, fuori da
            # ogni trial: verrebbero etichettate riposo per esclusione e non sono utilizzabili.
            keep = groups >= 0

            if DROP_BOUNDARY:
                # Una finestra di confine si riconosce dal fatto che e' etichettata riposo
                # ma il suo trial di maggioranza e' un trial attivo.
                active_trials = np.unique(groups[y != 1])
                keep &= ~((y == 1) & np.isin(groups, active_trials))

            per_run[run] = {
                "covs": covs,
                "ref": refs,
                "mask": keep,
                "y": y_binary[keep],
                "groups": groups[keep],
            }

        except Exception as e:
            print(f"Errore {subject} run {run}: {e}")

    if not all(r in per_run for r in runs):
        print(f"  Dati insufficienti: soggetto {subject} saltato\n")
        continue

    for test_index, test_run in enumerate(test_run_order):
        test_start = time.time()
        train_runs = [run for run in runs if run != test_run]

        # --- Riallineamento ---------------------------------------------------
        # Se il riferimento e' quello di training, si calcola una volta sola dalle covarianze
        # delle run di training messe insieme; altrimenti ogni run usa il proprio, gia' pronto.
        if RECENTER == "train":
            references = [
                reference_mean(np.concatenate([per_run[r]["covs"][b] for r in train_runs]))
                for b in range(len(BANDS))
            ]

        features = {}
        for run in runs:
            aligned = []
            for b in range(len(BANDS)):
                covs_b = per_run[run]["covs"][b]
                ref = references[b] if RECENTER == "train" else per_run[run]["ref"][b]
                aligned.append(recenter(covs_b, ref))

            # Solo dopo il riallineamento si tengono le finestre utilizzabili
            mask = per_run[run]["mask"]
            features[run] = to_tangent([a[mask] for a in aligned])

        X_train = np.concatenate([features[r] for r in train_runs])
        y_train = np.concatenate([per_run[r]["y"] for r in train_runs])
        groups_train = np.concatenate([per_run[r]["groups"] for r in train_runs])

        X_test = features[test_run]
        y_test = per_run[test_run]["y"]

        n_rest, n_active = int((y_train == 0).sum()), int((y_train == 1).sum())

        # --- Classificazione ---------------------------------------------------
        # Le due classi non sono bilanciate: il riposo occupa circa il doppio del tempo.
        # Senza indicazioni la LDA userebbe le frequenze osservate come probabilita' a priori e
        # favorirebbe il riposo; imponendole uguali si ottiene l'equivalente dello
        # class_weight='balanced' usato nella versione CSP.
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearDiscriminantAnalysis(solver="lsqr", priors=np.array([0.5, 0.5]))),
        ])

        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        grid = GridSearchCV(pipe, param_grid, cv=cv,
                            scoring="balanced_accuracy", n_jobs=-1)
        grid.fit(X_train, y_train, groups=groups_train)

        probs = grid.predict_proba(X_test)
        max_probs = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)

        threshold = START_THRESHOLD
        accepted_mask = max_probs >= threshold
        while np.sum(accepted_mask) < MIN_ACCEPTED_RATIO * len(y_test) and threshold > MIN_THRESHOLD:
            threshold -= 0.05
            accepted_mask = max_probs >= threshold

        accepted = int(np.sum(accepted_mask))
        total = len(y_test)
        discarded = total - accepted

        accuracy = balanced_accuracy_score(y_test[accepted_mask], predictions[accepted_mask])
        all_accuracy[i - first_person, test_index] = accuracy
        all_discarded[i - first_person, test_index] = 100 * discarded / total
        cm_sum += confusion_matrix(y_test[accepted_mask], predictions[accepted_mask])

        print(f"  Run test: {test_run} | Threshold: {threshold:.2f} | "
              f"Riallineamento: {RECENTER}")
        print(f"  Training: {n_rest} finestre di riposo / {n_active} di attivazione")
        print(f"  Campioni: {total} tot / {accepted} accettati / {discarded} scartati "
              f"({100*discarded/total:.1f}%)")
        print(f"  Balanced accuracy sugli accettati: {accuracy*100:.2f}% | "
              f"score interno: {grid.best_score_*100:.2f}% | shrinkage: {grid.best_params_}")
        print(f"  Tempo: {time.time() - test_start:.1f}s\n")

    patient_mean = np.nanmean(all_accuracy[i - first_person])
    patient_discarded = np.nanmean(all_discarded[i - first_person])
    print(f"  Paziente {subject}: {patient_mean*100:.2f}% con {patient_discarded:.1f}% di scarti "
          f"| Tempo: {time.time() - subject_start:.1f}s\n")

# --- Riepilogo ---------------------------------------------------------------

n_folds = int(np.count_nonzero(~np.isnan(all_accuracy)))
n_falliti = all_accuracy.size - n_folds
print("=" * 60)
print(f"Fold riusciti:          {n_folds}/{all_accuracy.size}"
      + (f" ({n_falliti} falliti, esclusi dalle medie)" if n_falliti else ""))
print(f"Accuratezza media:      {np.nanmean(all_accuracy)*100:.2f}%")
print(f"Deviazione standard:    {np.nanstd(all_accuracy)*100:.2f}%")
print(f"Campioni scartati:      {np.nanmean(all_discarded):.1f}% in media")
print(f"Matrice di confusione media:\n{(cm_sum / n_folds).astype(int)}")


Feature per finestra: 135 (3 bande x 9 canali)

Paziente 001
  Run test: 14 | Threshold: 0.75 | Riallineamento: per_run
  Training: 635 finestre di riposo / 420 di attivazione
  Campioni: 211 tot / 157 accettati / 54 scartati (25.6%)
  Balanced accuracy sugli accettati: 73.97% | score interno: 67.11% | shrinkage: {'clf__shrinkage': 0.5}
  Tempo: 9.6s

  Run test: 10 | Threshold: 0.70 | Riallineamento: per_run
  Training: 635 finestre di riposo / 420 di attivazione
  Campioni: 211 tot / 153 accettati / 58 scartati (27.5%)
  Balanced accuracy sugli accettati: 75.93% | score interno: 65.92% | shrinkage: {'clf__shrinkage': 0.5}
  Tempo: 4.8s

  Run test: 6 | Threshold: 0.70 | Riallineamento: per_run
  Training: 635 finestre di riposo / 420 di attivazione
  Campioni: 211 tot / 160 accettati / 51 scartati (24.2%)
  Balanced accuracy sugli accettati: 79.51% | score interno: 63.59% | shrinkage: {'clf__shrinkage': 0.5}
  Tempo: 0.6s

  Run test: 12 | Threshold: 0.70 | Riallineamento: per_run
  